# JEKOVA

In [ ]:
# IMPORTS ###########################################

import importlib
import warnings
warnings.filterwarnings("ignore")

#####################################################

import os
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt

from pxg import plot
importlib.reload(plot)

from pxg import Stop
from pxg import FS, MS
from pxg import EXG, Rids, Record
from pxg.rec import load_epi

from IPython.display import Markdown, display

%config InlineBackend.figure_format = "retina"

# show dataframes in full, no row/column truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# Visualization

def PlotPage(
    rec: Record, 
    page = 0, 
    offset = 0,
    marker = False,
    include = []
):
    res = plot.Page(
        rec, 
        page, 
        offset,

        SENSOR = False,
        SIGNAL = True,
        DECTOR = False,
        EXTEND = "JEKOVA",

        # LOW = -700,
        ZEROS = [0],

        # label = "BPM",
        angle = 0,

        rrqs  = False, 
        qsvl  = False, 
        
        punts = False,  
        trig  = False,
        onoff = False,
        letra = False,

        grid  = False,
        anref = False,
        simple = True,

        # simple = False,
        # marker = marker and page not in include,

        show = False,
    )

    if res:
        PON, POF = plot.Range(rec, page, offset)

        # plot.Signal(rec.Local[PON:POF] / MV, zero=-400, color="tab:blue", format="-", linewidth=0.6)
        # plot.Signal(rec.Cover[PON:POF] / MV, zero=-400, color="tab:orange", format="-", linewidth=0.5)

        # plot.Signal(rec.Maxim[PON:POF] / MV, zero=-400, color="tab:red", format="-", linewidth=0.8)
        # plot.Signal(rec.Minim[PON:POF] / MV, zero=-400, color="tab:red", format="-", linewidth=0.8)

        plot.Signal(rec.Digit[PON:POF] * 90, zero=-700, color="tab:gray", format="-", linewidth=0.5, fill=True, alpha=0.2)
        plot.Signal(rec.Class[PON:POF] * 90, zero=-700, color="tab:red", format="-", linewidth=0.5, fill=True, alpha=0.2)
    pass #if

    plot.Show()

    return res
pass #def


def PlotRecord(rec: Record, page: int | list[int] = -1, off = 0, marker = False, title = ""):
    offset = off*plot.FS
    display(Markdown(f"## {rec.DB.upper()} {rec.RID}\n---"))
    if title:
        display(Markdown(f"```text\n{title}\n```"))
    pass #if
    if isinstance(page, list):
        for p in page:
            PlotPage(rec, p, offset, marker = marker)
        pass #for
    elif page != -1:
        PlotPage(rec, page, offset, marker = marker)
    else:
        for page in range(0, (len(rec.Signal) + plot.CHUNK // 2) // plot.CHUNK, 1):
            PlotPage(rec, page, offset, marker = marker)
        pass #for
    pass #if
    return rec
pass #def


In [ ]:
LYN_WIND = 40 // MS

def LynnFilter(x: np.ndarray, w: int = LYN_WIND) -> np.ndarray:
    w -= w % 2

    a = np.array([1, -2, 1])
    b = np.zeros(w + 1)
    b[0] = 1
    b[w // 2] = -2
    b[w] = 1
    g = (len(b) // 2) ** 2
    shift = w // 2 - 1

    y = signal.lfilter(b, a, x) / g # type: ignore
    y = np.roll(y, -shift)
    return y
pass #def


FS_TARGET = 250    # Hz; every JEKOVA constant is defined for a 250 Hz sampling rate
HP_HZ = 1.0        # first-order high-pass cutoff (per stage)
LP_HZ = 30.0       # Butterworth low-pass cutoff
LP_ORDER = 2       # second-order Butterworth, as the paper specifies
NOTCH_HZ = 50.0    # powerline notch centre frequency (Sofia/European mains)
NOTCH_Q = 30.0     # notch quality factor (dimensionless); bandwidth = NOTCH_HZ / Q,
                   # so Q = 30 gives a ~1.7 Hz notch. The paper does not specify Q.

# Jekova band-pass (paper eq. 1) in integer form, carrying a gain of 16 like the
# Lynn filters. Paper equation, scaled by 16 to clear the fractions:
#     8*FS[i] = 14*FS[i-1] - 7*FS[i-2] + (S[i] - S[i-2])/2
#    16*FS[i] = 28*FS[i-1] - 14*FS[i-2] + (S[i] - S[i-2])
# Coefficients are integers; the peak gain at the 14.6 Hz centre is JEK_GAIN.
# Divide by JEK_GAIN to get unity peak gain. The Step 6 counts are all relative
# to a per-window max / mean / MD, so the cascade is unaffected by the scale.
JEK_B = [16, 0, -16]
JEK_A = [16, -28, 14]
JEK_GAIN = 16 // 4

def JakovaFilter(x, prep = True):
    y = np.asarray(x, dtype=float)

    if prep:
        ## High pass filter ###############
        hp_b, hp_a = signal.butter(1, HP_HZ / (0.5 * FS_TARGET), btype="highpass") # type: ignore
        y = signal.lfilter(hp_b, hp_a, y)
        y = signal.lfilter(hp_b, hp_a, y)

        ## Low pass filter ################
        lp_b, lp_a = signal.butter(LP_ORDER, LP_HZ / (0.5 * FS_TARGET), btype="lowpass") # type: ignore
        y = signal.lfilter(lp_b, lp_a, y)
        ###################################

        ## Notch filter ###################
        nt_b, nt_a = signal.iirnotch(NOTCH_HZ, NOTCH_Q, fs=FS_TARGET) # type: ignore
        y = signal.lfilter(nt_b, nt_a, y)
    else:
        y = LynnFilter(y)
    pass #if

    ## Jekova Equation ################
    k = signal.lfilter(JEK_B, JEK_A, y) / JEK_GAIN # type: ignore
    ###################################

    return y, np.array(k)
pass #def

# calculate Jekova counts in a 1 s segment of the band-pass output (paper step 5-6)
def Counts(seg: np.ndarray) -> tuple[int, int, int]:
    # step 5: the counts are defined on the absolute (rectified) filter output
    abs_fs = np.abs(np.asarray(seg, dtype=float))

    smax  = np.max(abs_fs)
    smean = np.mean(abs_fs)
    md    = np.mean(np.abs(abs_fs - smean))   # mean absolute deviation, not median

    # step 6: number of samples falling in each amplitude band
    c1 = int(np.sum(abs_fs >= smax / 2))                                # 0.5*smax .. smax
    c2 = int(np.sum(abs_fs >= smean))                                   # smean    .. smax
    c3 = int(np.sum((abs_fs >= smean - md) & (abs_fs <= smean + md)))   # smean +- MD

    return c1, c2, c3
pass #def
    

COLUMNS = ["db", "rid", "pon", "pof", "vfb", "c1", "c2", "c3"]

def Process(rec: Record) -> tuple[list[dict], list[dict], list[dict], np.ndarray]:
    _, jek = JakovaFilter(rec.Point, False)
    rec.Jekova = jek

    vfib = np.zeros(len(rec.Point), dtype=int)
    for epi in rec.RefEpi:
        if epi.Name in ["#VT", "VFL", "VF", "WF"]:
            vfib[epi.Time:epi.End+1] = 1
        pass #if
    pass #for
    rec.RefVFB = vfib

    rows = []
    for s in range(len(rec.Point) // FS):
        pon = s * FS
        pof = pon + FS
        seg = vfib[pon:pof]
        vfb = int(np.sum(seg))
        c1, c2, c3 = Counts(jek[pon:pof])
        rows.append(dict(db=rec.DB, rid=rec.RID, pon=pon, pof=pof, vfb=vfb, c1=c1, c2=c2, c3=c3))
    pass #for

    row4 = []
    for s in range(len(rows) - 4):
        vfb = sum(rows[s+i]["vfb"] for i in range(4))
        c1  = sum(rows[s+i]["c1"]  for i in range(4))
        c2  = sum(rows[s+i]["c2"]  for i in range(4))
        c3  = sum(rows[s+i]["c3"]  for i in range(4))
        row4.append(dict(db=rec.DB, rid=rec.RID, pon=rows[s]["pon"], pof=rows[s+3]["pof"], vfb=vfb, c1=c1, c2=c2, c3=c3))
    pass #for

    row8 = []
    for s in range(len(rows) - 8):
        vfb = sum(rows[s+i]["vfb"] for i in range(8))
        c1  = sum(rows[s+i]["c1"]  for i in range(8))
        c2  = sum(rows[s+i]["c2"]  for i in range(8))
        c3  = sum(rows[s+i]["c3"]  for i in range(8))
        row8.append(dict(db=rec.DB, rid=rec.RID, pon=rows[s]["pon"], pof=rows[s+7]["pof"], vfb=vfb, c1=c1, c2=c2, c3=c3))
    pass #for

    return rows, row4, row8, rec.RefVFB
pass #def

def Prepare(db: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows = []
    row4 = []
    row8 = []

    # mkdir ../work/vfib/{db}
    os.makedirs(f"../work/vfib/{db}", exist_ok=True)

    rids = Rids(db)
    for rid in rids:
        rec = Record(db, rid)
        rs, r4, r8, vfb = Process(rec)
        rows.extend(rs)
        row4.extend(r4)
        row8.extend(r8)

        # save the per-record VFB mask as a compact binary (uint8)
        np.save(f"../work/vfib/{db}/{rid}.ref.npy", vfb.astype(np.uint8))
    pass #for
    dfs = pd.DataFrame(rows, columns=COLUMNS)
    df4 = pd.DataFrame(row4, columns=COLUMNS)
    df8 = pd.DataFrame(row8, columns=COLUMNS)

    # Save to work/vfib/{db}
    dfs.to_csv(f"../work/vfib/{db}_dfs.tsv", index=False, sep="\t")
    df4.to_csv(f"../work/vfib/{db}_df4.tsv", index=False, sep="\t")
    df8.to_csv(f"../work/vfib/{db}_df8.tsv", index=False, sep="\t")

    return dfs, df4, df8
pass #def

In [ ]:
def PlotDatabase(db: str):
    # rids = EXG(db, learn = True, cfm = True, devx = False, wrx = False)
    rids = Rids(db)
    for rid in rids:
        rec = Record(db, rid)
        rec.DetEpi = load_epi(f"../work/output/pxg/anv/{db}/{rid}.anv", False)
        _, rec.Jekova = JakovaFilter(rec.Point, False)
        for page in range(0, (len(rec.Signal) + plot.CHUNK // 2) // plot.CHUNK, 1):
            PON, POF = plot.Range(rec, page, 0)
            cc = 0
            for epi in rec.RefEpi:
                if epi.End < PON or epi.Time >= POF: continue
                if epi.Name in ["#VT", "VFL", "VF", "WF"]: cc+=1
            pass #for
            for epi in rec.DetEpi:
                if epi.End < PON or epi.Time >= POF: continue
                if epi.Name in ["#VT", "VFL", "VF", "WF"]: cc+=1
            pass #for
            if cc == 0: continue
            PlotRecord(rec, page = [page], off = 0, marker = False, title = "")
        pass #for
    pass #for
pass #def

In [ ]:
DATABASES = ["mitdb", "ahadb", "cudb", "edb", "vfdb"]

for db in DATABASES:
    print(db)
    #Prepare(db)
pass #for

In [ ]:
# Evaluate the published Jekova cascade, thresholds scaled to the window length.

EPOCH = 10 * FS          # the paper's 10 s epoch at 250 Hz = 2500 samples
FRAC = 0.85

def LoadDatabase(db: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Reload the per-second / 4 s / 8 s frames written by ProcessDatabase."""
    dfs = pd.read_csv(f"../work/vfib/{db}_dfs.tsv", sep="\t")
    df4 = pd.read_csv(f"../work/vfib/{db}_df4.tsv", sep="\t")
    df8 = pd.read_csv(f"../work/vfib/{db}_df8.tsv", sep="\t")
    return dfs, df4, df8
pass #def

# Published thresholds (paper section 2.2.5), valid for a 2500-sample epoch. Every
# count is a number of samples, so all of them scale linearly with the window
# length; the ratio Count1*Count2/Count3 scales linearly too, since N*N/N = N.
JEKOVA_THRESHOLDS = dict(C1_LO=250, C1_HI=400, C2_LO=600, C2_HI=950, C2_MAX=1100, RATIO=210)

def Thresholds(span: int, t: dict = JEKOVA_THRESHOLDS) -> dict:
    """Scale the published 10 s thresholds to a window of ``span`` samples."""
    k = span / EPOCH
    return {name: value * k for name, value in t.items()}
pass #def

def Classify(df: pd.DataFrame, t: dict = JEKOVA_THRESHOLDS) -> np.ndarray:
    c1 = df["c1"].to_numpy(dtype=float)
    c2 = df["c2"].to_numpy(dtype=float)
    c3 = df["c3"].to_numpy(dtype=float)

    rr = c1 * c2 / np.where(c3 > 0, c3, np.nan)

    r1 = (c1 <  t["C1_LO"]) & (c2 > t["C2_HI"]) & (rr < t["RATIO"])
    r2 = (c1 >= t["C1_LO"]) & (c1 < t["C1_HI"]) & (c2 < t["C2_LO"]) & (rr < t["RATIO"])
    r3 = (c1 >= t["C1_LO"]) & (c2 > t["C2_HI"])
    r4 = (c2 >= t["C2_MAX"])

    out = np.full(len(df), "?", dtype="<U1")
    for rule, label in [(r1, "N"), (r2, "N"), (r3, "S"), (r4, "S")]:   # first match wins
        out = np.where((out == "?") & rule, label, out)
    pass #for
    return out
pass #def

def Rates(tp: int, fn: int, fp: int, tn: int) -> dict:
    """Confusion counts and derived rates, shared by the window and sample tables."""
    return dict(
        TP=tp, FN=fn, FP=fp, TN=tn,
        SPC = round(100 * tn / (tn + fp), 2) if tn + fp else float("nan"),
        SEN = round(100 * tp / (tp + fn), 2) if tp + fn else float("nan"),
        PPV = round(100 * tp / (tp + fp), 2) if tp + fp else float("nan"),
        F1  = round(100 * 2 * tp / (2 * tp + fp + fn), 2) if 2 * tp + fp + fn else float("nan"),
    )
pass #def

def MetricsWin(call, shock, nonsh, mixed) -> dict:
    """Window-level confusion for one boolean slice of a database."""
    tp = int(((call == "S") & shock).sum())
    fn = int(((call != "S") & shock).sum())
    fp = int(((call == "S") & nonsh).sum())
    tn = int(((call != "S") & nonsh).sum())
    return dict(nonsh=int((nonsh & ~mixed).sum()), mixed=int(mixed.sum()), shock=int(shock.sum()), **Rates(tp, fn, fp, tn))
pass #def

def EvaluateWin(df: pd.DataFrame, frac: float = FRAC, span: int = FS * 8) -> pd.DataFrame:
    """Per-record window rows (one per rid) plus a TOTAL row."""
    k = Thresholds(span, JEKOVA_THRESHOLDS)
    call = Classify(df, k)

    # frac is a cut, not a band: every window lands in exactly one class.
    shock = (df["vfb"] >= frac * span).to_numpy()
    nonsh = ~shock
    # windows straddling an episode edge: some VF present but under the cut,
    # scored as negatives, reported apart from the clear negatives
    mixed = (df["vfb"].to_numpy() > 0) & nonsh

    db  = df["db"].iloc[0]
    rid = df["rid"].to_numpy()

    rows = []
    for r in pd.unique(rid):
        m = rid == r
        rows.append({"db": db, "rid": str(r), **MetricsWin(call[m], shock[m], nonsh[m], mixed[m])})
    pass #for
    rows.append({"db": db, "rid": "TOTAL", **MetricsWin(call, shock, nonsh, mixed)})
    return pd.DataFrame(rows)
pass #def

def EvaluateSmp(df: pd.DataFrame, frac: float = FRAC, span: int = FS * 8) -> pd.DataFrame:
    """Per-record SAMPLE rows (same columns as EvaluateWin) plus a TOTAL row.

    The window decisions are mapped back to a per-sample prediction mask (a sample
    is shockable if any covering window was called "S"), compared against the
    per-sample reference, and the mask saved as {db}/{rid}.vfb.npy.
    """
    k = Thresholds(span, JEKOVA_THRESHOLDS)
    call = Classify(df, k)

    db  = df["db"].iloc[0]
    rid = df["rid"].to_numpy()
    pon = df["pon"].to_numpy()
    pof = df["pof"].to_numpy()

    rows = []
    tot = [0, 0, 0, 0]                    # pooled sample TP, FN, FP, TN for TOTAL
    for r in pd.unique(rid):
        m = rid == r

        ref  = np.load(f"../work/vfib/{db}/{r}.ref.npy").astype(bool)
        pred = np.zeros(len(ref), dtype=bool)
        for a, b, cl in zip(pon[m], pof[m], call[m]):
            if cl == "S":
                pred[a:b] = True
            pass #if
        pass #for
        np.save(f"../work/vfib/{db}/{r}.vfb.npy", pred.astype(np.uint8))

        # Save as detected annotations
        os.makedirs(f"../work/output/pxg/anv/{db}", exist_ok=True)
        with open(f"../work/output/pxg/anv/{db}/{r}.anv", "w") as f:
            prev = -1
            for ix in range(len(pred)):
                p = int(pred[ix])
                if p == prev: continue
                if p == 0:
                    f.write(f"{ix},+,(NF\n")
                else:
                    f.write(f"{ix},+,(WF\n")
                pass #if
                prev = p
            pass #for
            f.write(f"{len(pred)},+,(END\n")
        pass #with

        tp = int((pred & ref).sum());  fn = int((~pred & ref).sum())
        fp = int((pred & ~ref).sum()); tn = int((~pred & ~ref).sum())
        for i, v in enumerate((tp, fn, fp, tn)):
            tot[i] += v
        pass #for
        rows.append({"db": db, "rid": str(r), "nonsh": tn + fp, "mixed": 0, "shock": tp + fn, **Rates(tp, fn, fp, tn)})
    pass #for
    tp, fn, fp, tn = tot
    rows.append({"db": db, "rid": "TOTAL", "nonsh": tn + fp, "mixed": 0, "shock": tp + fn, **Rates(tp, fn, fp, tn)})
    return pd.DataFrame(rows)
pass #def

In [17]:
win, smp = [], []
for db in DATABASES:
    _, _, df8 = LoadDatabase(db)
    rw = EvaluateWin(df8)
    rs = EvaluateSmp(df8)
    if db in ["mitdb", "ahadb", "cudb"]:
        display(Markdown(f"### {db} — per window"));   display(rw)
        # display(Markdown(f"### {db} — per sample"));   display(rs)
    pass #if
    win.append(rw[rw["rid"] == "TOTAL"])
    smp.append(rs[rs["rid"] == "TOTAL"])
pass #for

display(Markdown("## Summary — per sample"))
display(pd.concat(smp, ignore_index=True))

# summary: the TOTAL row of every database, window vs sample
display(Markdown("## Summary — per window"))
display(pd.concat(win, ignore_index=True))

### mitdb — per window

,db,rid,nonsh,mixed,shock,TP,FN,FP,TN,SPC,SEN,PPV,F1
0,mitdb,100,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
1,mitdb,101,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
2,mitdb,102,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
3,mitdb,103,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
4,mitdb,104,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
5,mitdb,105,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
6,mitdb,106,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
7,mitdb,107,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
8,mitdb,108,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN
9,mitdb,109,1794,0,0,0,0,0,1794,100.00,NaN,NaN,NaN


### ahadb — per window

,db,rid,nonsh,mixed,shock,TP,FN,FP,TN,SPC,SEN,PPV,F1
0,ahadb,1201,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
1,ahadb,1202,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
2,ahadb,1203,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
3,ahadb,1204,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
4,ahadb,1206,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
5,ahadb,1207,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
6,ahadb,1208,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
7,ahadb,1209,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN
8,ahadb,1210,1777,0,0,0,0,16,1761,99.10,NaN,0.00,0.00
9,ahadb,2201,1777,0,0,0,0,0,1777,100.00,NaN,NaN,NaN


### cudb — per window

,db,rid,nonsh,mixed,shock,TP,FN,FP,TN,SPC,SEN,PPV,F1
0,cudb,cu01,207,6,286,285,1,0,213,100.00,99.65,100.00,99.82
1,cudb,cu02,499,0,0,0,0,5,494,99.00,NaN,0.00,0.00
2,cudb,cu03,458,7,34,34,0,1,464,99.78,100.00,97.14,98.55
3,cudb,cu04,195,54,250,247,3,30,219,87.95,98.80,89.17,93.74
4,cudb,cu05,403,14,82,74,8,1,416,99.76,90.24,98.67,94.27
5,cudb,cu06,346,27,126,117,9,8,365,97.86,92.86,93.60,93.23
6,cudb,cu07,175,6,318,297,21,0,181,100.00,93.40,100.00,96.59
7,cudb,cu08,419,7,73,61,12,6,420,98.59,83.56,91.04,87.14
8,cudb,cu09,434,13,52,52,0,31,416,93.06,100.00,62.65,77.04
9,cudb,cu10,309,7,183,130,53,12,304,96.20,71.04,91.55,80.00


## Summary — per sample

,db,rid,nonsh,mixed,shock,TP,FN,FP,TN,SPC,SEN,PPV,F1
0,mitdb,TOTAL,21591126,0,35754,33045,2709,1705,21589421,99.99,92.42,95.09,93.74
1,ahadb,TOTAL,33915108,0,1355548,1324009,31539,50241,33864867,99.85,97.67,96.34,97.00
2,cudb,TOTAL,3495017,0,949143,799878,149265,236872,3258145,93.22,84.27,77.15,80.56
3,edb,TOTAL,161832960,0,0,0,0,35500,161797460,99.98,NaN,0.00,0.00
4,vfdb,TOTAL,10937863,0,596473,547391,49082,1805359,9132504,83.49,91.77,23.27,37.12


## Summary — per window

,db,rid,nonsh,mixed,shock,TP,FN,FP,TN,SPC,SEN,PPV,F1
0,mitdb,TOTAL,85932,69,111,101,10,3,85998,100.00,90.99,97.12,93.95
1,ahadb,TOTAL,134939,98,5346,5086,260,121,134916,99.91,95.14,97.68,96.39
2,cudb,TOTAL,13432,540,3493,2675,818,577,13395,95.87,76.58,82.26,79.32
3,edb,TOTAL,646560,0,0,0,0,61,646499,99.99,NaN,0.00,0.00
4,vfdb,TOTAL,42994,942,2022,1918,104,6263,37673,85.75,94.86,23.44,37.60


In [ ]:
PlotDatabase("mitdb")

In [ ]:
Stop()

# Decision tree on the counts c1, c2, c3
#
# A shallow tree just learns count thresholds (what the cascade does by hand), and
# its rules are readable. Evaluated two ways: in-sample (comparable to the manual
# thresholds, which were also tuned in-sample) and grouped cross-validation with
# GroupKFold by record, so windows from one record never leak between folds. The
# gap between the two is the per-record drift.

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GroupKFold, cross_val_predict

TREE_DBS = ["mitdb", "ahadb", "cudb", "vfdb"]   # edb has no shockable windows

frames = []
for db in TREE_DBS:
    _, _, df8 = LoadDatabase(db)
    frames.append(df8)
pass #for
D = pd.concat(frames, ignore_index=True)

span = FS * 8
X = D[["c1", "c2", "c3"]].to_numpy(float)
y = (D["vfb"].to_numpy() >= FRAC * span).astype(int)
groups = (D["db"] + "/" + D["rid"].astype(str)).to_numpy()   # one group per record

def tree_report(y_true, y_pred, mask, name) -> dict:
    yy, pp = y_true[mask], y_pred[mask]
    tp = int(((pp == 1) & (yy == 1)).sum()); fp = int(((pp == 1) & (yy == 0)).sum())
    fn = int(((pp == 0) & (yy == 1)).sum()); tn = int(((pp == 0) & (yy == 0)).sum())
    return dict(set=name, shock=int(yy.sum()), TP=tp, FN=fn, FP=fp, TN=tn,
                SPC=round(100 * tn / (tn + fp), 2) if tn + fp else float("nan"),
                SEN=round(100 * tp / (tp + fn), 2) if tp + fn else float("nan"),
                PPV=round(100 * tp / (tp + fp), 2) if tp + fp else float("nan"),
                F1 =round(100 * 2 * tp / (2 * tp + fp + fn), 2) if 2 * tp + fp + fn else float("nan"))
pass #def

clf = DecisionTreeClassifier(max_depth=4, random_state=0)
insample = clf.fit(X, y).predict(X)
crossval = cross_val_predict(clf, X, y, groups=groups, cv=GroupKFold(5))

for tag, pred in [("in-sample", insample), ("grouped-CV", crossval)]:
    rows = [tree_report(y, pred, (D["db"] == db).to_numpy(), db) for db in TREE_DBS]
    rows.append(tree_report(y, pred, np.ones(len(y), bool), "POOLED"))
    display(Markdown(f"### Decision tree (depth 4) - {tag}"))
    display(pd.DataFrame(rows))
pass #for

# the learned thresholds
print(export_text(clf.fit(X, y), feature_names=["c1", "c2", "c3"], max_depth=3))